In [2]:
from dashscope.acli.tools.checkpoint import snapshot
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv
from langgraph.checkpoint.postgres import PostgresSaver
from psycopg.rows import dict_row
from psycopg_pool import ConnectionPool

from app.core.config import settings

load_dotenv()

#==================1. 初始化连接池===================
# 创建连接池。实际开发中将ConnectionPool换成异步的AsyncConnectionPool
#https://www.psycopg.org/psycopg3/docs/api/pool.html#psycopg_pool.AsyncConnectionPool
#https://www.psycopg.org/psycopg3/docs/api/connections.html#psycopg.Connection
checkpoint_pool = ConnectionPool(
    conninfo=settings.db.checkpoint_url,
    min_size=1,  # 池最小连接数
    max_size=5,  # 池最大连接数
    kwargs={
        "autocommit": True,  # 自动提交事务
        "prepare_threshold": 0,  # 不做预编译sql
        "row_factory": dict_row,  # 查询结果以dict返回
    },
    open=False,  # 不自动创建连接
)
# 初始化连接
checkpoint_pool.open()
# 等待连接池就绪
checkpoint_pool.wait()

#==================2. 初始化PostgresSaver===================
# 初始化checkpointer。实际开发中将 PostgresSaver 换成异步的 AsyncPostgresSaver
# https://reference.langchain.com/python/langgraph.checkpoint.postgres/aio/AsyncPostgresSaver
checkpointer = PostgresSaver(checkpoint_pool)
# 自动建表（与checkpointer存储有关的表）
checkpointer.setup()

model = init_chat_model(model = "deepseek-flash",extra_body={"thinking":{"type":"disabled"}})

agent = create_agent(model = model,tools =[],checkpointer = checkpointer)

#验证步骤：
print("--------------第一次对话--------------")
response = agent.invoke(
     {"messages":[HumanMessage(content="我是小团团，我在抖音直播影刃")]},
     config={"configurable":{"thread_id":"day06-test-checkpointer"}})
print(response["messages"][-1].content)

print("--------------第二次对话--------------")
#   第2步：调用智能体，发送消息：我是谁？我喜欢什么   然后查看模型的回答是否具有记忆
response = agent.invoke(
    {"messages":[HumanMessage(content="我是谁？我喜欢什么")]},
    config={"configurable":{"thread_id":"day06-test-checkpointer"}}
)
print(response['messages'][-1].content)

print("=================查询聊天记录=============================")
snapshot = agent.get_state({"configurable":{"thread_id":"day06-test-checkpointer"}})
message_list = snapshot.values.get("messages",[])
print(message_list)


--------------第一次对话--------------
你好呀小团团！🎉 看到你在抖音直播《影刃》，太棒了！《影刃》这款游戏确实很适合直播，动作打击感和画面表现力都很适合和观众互动。

作为AI助手，我可以帮你做这些事：

1. **直播话术建议**：帮你设计开场白、互动话题、下播感谢词等
2. **游戏内容策划**：帮你规划直播环节（比如挑战任务、水友赛、抽奖互动）
3. **短视频脚本**：把直播高光时刻剪成短视频的文案和标题
4. **粉丝互动点子**：设计弹幕互动玩法、粉丝专属福利
5. **数据分析建议**：帮你分析什么时间段直播效果好、哪些内容互动率高

你现在直播遇到什么问题了吗？还是想聊聊直播内容怎么策划？或者需要我帮你写点什么直播相关的文案？尽管说～ 😊
--------------第二次对话--------------
哈哈，根据你刚才告诉我的信息：

**你是谁：**
你是**小团团**，在**抖音**直播，直播的游戏是**《影刃》**。

**你喜欢什么：**
你目前只提到了直播《影刃》这件事，所以我只能推断——你至少喜欢（或正在做）**直播游戏《影刃》** 这件事 😄

至于你更深层的喜好（比如喜欢的英雄/武器、喜欢的直播风格、喜欢的零食、喜欢的梗……），你还没告诉我呢！

要不你多透露一点？比如：
- 你在《影刃》里最爱玩哪个角色/职业？
- 你直播时最喜欢和粉丝玩什么互动？
- 直播之外你喜欢做什么？

这样我就能更了解你，帮你把直播做得更带劲啦～ 🎮✨
